In [1]:
import sys
from pathlib import Path

import ee
import pandas as pd

# tests/backtests.ipynb → RozviDrought project root
PROJECT_ROOT = Path.cwd().resolve()

# If notebook is launched from tests/, move one level up.
if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

EE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine initialized with project: {EE_PROJECT}")
except Exception as exc:
    print("Earth Engine not initialized. Running authentication...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT}")

print("Project root:", PROJECT_ROOT)

Earth Engine initialized with project: august-analyze
Project root: C:\Projects\Infer RozviDrought\RozviDrought


In [2]:
# Cell 2: Load Zimbabwe administrative polygons from Earth Engine

ADMIN_LEVEL = 2
GAUL_ASSET = f"FAO/GAUL/2015/level{ADMIN_LEVEL}"

admin_fc = (
    ee.FeatureCollection(GAUL_ASSET)
    .filter(ee.Filter.eq("ADM0_NAME", "Zimbabwe"))
)

admin_count = admin_fc.size().getInfo()

first_admin = admin_fc.first().toDictionary().getInfo()

print(f"Loaded Zimbabwe ADM{ADMIN_LEVEL} polygons from:", GAUL_ASSET)
print("Admin polygon count:", admin_count)
print("Example properties:")
first_admin

Loaded Zimbabwe ADM2 polygons from: FAO/GAUL/2015/level2
Admin polygon count: 62
Example properties:


{'ADM0_CODE': 271,
 'ADM0_NAME': 'Zimbabwe',
 'ADM1_CODE': 3436,
 'ADM1_NAME': 'Harare',
 'ADM2_CODE': 68807,
 'ADM2_NAME': 'Chitungwiza',
 'DISP_AREA': 'NO',
 'EXP2_YEAR': 3000,
 'STATUS': 'Member State',
 'STR2_YEAR': 2006,
 'Shape_Area': 0.00407260399874,
 'Shape_Leng': 0.34197890796}

In [3]:
# Cell 3: Pull Zimbabwe ADM2 polygons from Earth Engine into local records

admin_features = admin_fc.getInfo()["features"]

admin_records = []

for feature in admin_features:
    props = feature["properties"]
    geom = feature["geometry"]

    admin_records.append({
        "adm0_name": props.get("ADM0_NAME"),
        "adm1_name": props.get("ADM1_NAME"),
        "adm2_name": props.get("ADM2_NAME"),
        "adm2_code": props.get("ADM2_CODE"),
        "geometry": geom,
    })

admin_df = pd.DataFrame(admin_records)

print("Admin records loaded:", len(admin_df))
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())

Admin records loaded: 62
             adm1_name    adm2_name  adm2_code
0               Harare  Chitungwiza      68807
1               Harare       Harare      68809
2  Mashonaland Central       Guruve      68808
3  Mashonaland Central        Mbire      68811
4           Manicaland       Makoni      33056


In [4]:
# Cell 4: Convert ADM2 GeoJSON geometries to Shapely geometries

from shapely.geometry import shape

admin_df["shapely_geometry"] = admin_df["geometry"].apply(shape)

print("Converted ADM2 geometries to Shapely.")
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())
print("Example geometry type:", admin_df.loc[0, "shapely_geometry"].geom_type)

Converted ADM2 geometries to Shapely.
             adm1_name    adm2_name  adm2_code
0               Harare  Chitungwiza      68807
1               Harare       Harare      68809
2  Mashonaland Central       Guruve      68808
3  Mashonaland Central        Mbire      68811
4           Manicaland       Makoni      33056
Example geometry type: Polygon


In [5]:
# Cell 5: Load master model inputs and initialize polygon inference service

from app.services.polygon_inference_service import PolygonInferenceService

WORKSPACE_DIR = PROJECT_ROOT.parent

MASTER_PATH = (
    WORKSPACE_DIR
    / "data"
    / "master_inputs"
    / "master_inputs_long_198001_205012.parquet"
)

print("Using dataset:", MASTER_PATH)

master_df = pd.read_parquet(MASTER_PATH)

service = PolygonInferenceService(master_df=master_df)

print("Master rows:", len(master_df))
print("Master columns:", len(master_df.columns))
print("PolygonInferenceService initialized.")

Using dataset: C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Master rows: 38957625
Master columns: 13
PolygonInferenceService initialized.


In [6]:
# Cell 7: Hardcode drought timeline and expand to monthly backtest targets

DROUGHT_EVENTS = [
    {"event_id": 1, "period": "1902-1903", "season": "1902/03", "start_year": 1902, "end_year": 1903, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 2, "period": "1911-1912", "season": "1911/12", "start_year": 1911, "end_year": 1912, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 3, "period": "1921-1922", "season": "1921/22", "start_year": 1921, "end_year": 1922, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 4, "period": "1932-1933", "season": "1932/33", "start_year": 1932, "end_year": 1933, "duration_months": 12, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 5, "period": "1946-1947", "season": "1946/47", "start_year": 1946, "end_year": 1947, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 6, "period": "1967-1968", "season": "1967/68", "start_year": 1967, "end_year": 1968, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 7, "period": "1972-1973", "season": "1972/73", "start_year": 1972, "end_year": 1973, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 8, "period": "1982-1984", "season": "1982-84", "start_year": 1982, "end_year": 1984, "duration_months": 24, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 9, "period": "1986-1987", "season": "1986/87", "start_year": 1986, "end_year": 1987, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 10, "period": "1991-1992", "season": "1991/92", "start_year": 1991, "end_year": 1992, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
    {"event_id": 11, "period": "1994-1995", "season": "1994/95", "start_year": 1994, "end_year": 1995, "duration_months": 10, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 12, "period": "1997-1998", "season": "1997/98", "start_year": 1997, "end_year": 1998, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 13, "period": "2001-2002", "season": "2001/02 + 2002/03", "start_year": 2001, "end_year": 2002, "duration_months": 18, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 14, "period": "2012-2013", "season": "2012/13", "start_year": 2012, "end_year": 2013, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 15, "period": "2015-2016", "season": "2015/16", "start_year": 2015, "end_year": 2016, "duration_months": 12, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 16, "period": "2018-2019", "season": "2018/19", "start_year": 2018, "end_year": 2019, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 17, "period": "2023-2024", "season": "2023/24", "start_year": 2023, "end_year": 2024, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
]

yyyymm_col = "yyyymm" if "yyyymm" in master_df.columns else "date"

available_months = set(master_df[yyyymm_col].astype(int).unique())

monthly_rows = []

for event in DROUGHT_EVENTS:
    for year in range(event["start_year"], event["end_year"] + 1):
        for month in range(1, 13):
            yyyymm = int(f"{year}{month:02d}")

            monthly_rows.append({
                **event,
                "year": year,
                "month": month,
                "yyyymm": yyyymm,
                "available_in_master": yyyymm in available_months,
            })

backtest_months_df = pd.DataFrame(monthly_rows)

available_backtest_months_df = backtest_months_df[
    backtest_months_df["available_in_master"]
].reset_index(drop=True)

print("Total expanded event-month rows:", len(backtest_months_df))
print("Available event-month rows in master_df:", len(available_backtest_months_df))
print("First available rows:")
print(available_backtest_months_df.head(12))

Total expanded event-month rows: 420
Available event-month rows in master_df: 252
First available rows:
    event_id     period   season  start_year  end_year  duration_months  \
0          8  1982-1984  1982-84        1982      1984               24   
1          8  1982-1984  1982-84        1982      1984               24   
2          8  1982-1984  1982-84        1982      1984               24   
3          8  1982-1984  1982-84        1982      1984               24   
4          8  1982-1984  1982-84        1982      1984               24   
5          8  1982-1984  1982-84        1982      1984               24   
6          8  1982-1984  1982-84        1982      1984               24   
7          8  1982-1984  1982-84        1982      1984               24   
8          8  1982-1984  1982-84        1982      1984               24   
9          8  1982-1984  1982-84        1982      1984               24   
10         8  1982-1984  1982-84        1982      1984               24

In [ ]:
# Cell 8: Safe ADM2 drought backtest smoke test

import gc
import time

SCENARIO = "historical"
MODEL = "hybrid"

# Keep this tiny for the smoke test.
MAX_ADMINS = 1
MAX_MONTHS = 1

# Prefer small ADM2 polygons first to reduce pixel load.
admin_test_df = admin_df.copy()
admin_test_df["area_deg2"] = admin_test_df["shapely_geometry"].apply(lambda g: g.area)
admin_test_df = admin_test_df.sort_values("area_deg2").head(MAX_ADMINS).reset_index(drop=True)

# Use latest available drought months first because recent data is more likely complete.
month_test_df = (
    available_backtest_months_df
    .sort_values("yyyymm", ascending=False)
    .head(MAX_MONTHS)
    .reset_index(drop=True)
)

print("Safe smoke test scope")
print("Admins tested:", len(admin_test_df))
print("Months tested:", len(month_test_df))
print("Selected admin:")
print(admin_test_df[["adm1_name", "adm2_name", "adm2_code", "area_deg2"]])
print("Selected month:")
print(month_test_df[["event_id", "period", "yyyymm", "timeline_severity", "severity_label"]])

smoke_result = None
smoke_error = None

for admin_i, admin_row in admin_test_df.iterrows():
    for month_i, event_row in month_test_df.iterrows():
        started = time.time()

        print(
            f"\nRunning smoke inference "
            f"admin {admin_i + 1}/{len(admin_test_df)} | "
            f"month {month_i + 1}/{len(month_test_df)} | "
            f"{admin_row['adm2_name']} | {int(event_row['yyyymm'])}"
        )

        try:
            result = service.infer_polygon(
                geometry=admin_row["shapely_geometry"],
                scenario=SCENARIO,
                yyyymm=int(event_row["yyyymm"]),
                model=MODEL,
            )

            smoke_result = {
                "admin": admin_row[["adm1_name", "adm2_name", "adm2_code"]].to_dict(),
                "event": event_row[[
                    "event_id",
                    "period",
                    "season",
                    "yyyymm",
                    "timeline_severity",
                    "severity_label",
                ]].to_dict(),
                "summary": result.summary,
                "cell_result_count": len(result.cell_results),
            }

            print("Success.")
            print("Elapsed seconds:", round(time.time() - started, 2))
            break

        except Exception as exc:
            smoke_error = {
                "admin": admin_row[["adm1_name", "adm2_name", "adm2_code"]].to_dict(),
                "yyyymm": int(event_row["yyyymm"]),
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

            print("Failed.")
            print("Elapsed seconds:", round(time.time() - started, 2))
            print("Error type:", smoke_error["error_type"])
            print("Error:", smoke_error["error"])

        finally:
            gc.collect()

    if smoke_result is not None:
        break

if smoke_result is None:
    print("\nNo successful smoke result.")
    print("Last error:")
    print(smoke_error)
else:
    print("\nSmoke result:")
    print(smoke_result)